# RobotMath2030 — Diffusion Policy 2D (Ch.07)

Multimodal trajectories: mean regression vs diffusion.

Full chapter: [07_diffusion_policy_2d](../chapters/07_diffusion_policy_2d/)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    !git clone -q https://github.com/rsasaki0109/RobotMath2030.git 2>/dev/null || true
    %cd RobotMath2030
except ImportError:
    root = Path.cwd()
    if not (root / 'robotmath').exists() and (root.parent / 'robotmath').exists():
        %cd ..

!pip install -q -e ".[torch]"
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from miniworlds.two_path_world import START, GOAL, collision_rate, generate_demonstrations
from robotmath.diffusion import DiffusionPolicy2D, TrainConfig, predict_mean_regression, train_mean_regression
from robotmath.viz.plot_trajectory import draw_trajectories, draw_two_path_world

horizon = 24
demos, cond = generate_demonstrations(n_per_mode=32, horizon=horizon, seed=0)
test_cond = np.concatenate([START, GOAL])[None, :]

mean_model = train_mean_regression(demos, cond, horizon=horizon, epochs=60, seed=0)
mean_pred = predict_mean_regression(mean_model, test_cond, horizon=horizon)

cfg = TrainConfig(horizon=horizon, timesteps=20, epochs=80, seed=0)
policy = DiffusionPolicy2D(cfg)
policy.fit(demos, cond)
samples = policy.sample(test_cond, n_samples=16)

print(f'Mean regression collision: {collision_rate(mean_pred):.0%}')
print(f'Diffusion sample collision: {collision_rate(samples):.0%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
draw_two_path_world(axes[0])
draw_trajectories(axes[0], mean_pred, color='C3', linewidth=2.5)
axes[0].set_title('Mean regression → collision')
draw_two_path_world(axes[1])
draw_trajectories(axes[1], samples, color='C0', alpha=0.6)
axes[1].set_title('Diffusion → multimodal paths')
plt.tight_layout()
plt.show()